In [ ]:
run_on_colab = False
if run_on_colab:
    !pip install tiktoken
    %rm -rf aptax
    !git clone https://github.com/antodima/aptax.git
    %cd aptax/
else:
    %cd ..

In [ ]:
from collections import defaultdict

import json
from tqdm import tqdm
from pathlib import Path

from transformers import AutoTokenizer

import jax
import jax.numpy as jnp
from jax.sharding import SingleDeviceSharding

import flax.nnx as nnx
from flax.training.early_stopping import EarlyStopping

import optax
import orbax
from orbax import checkpoint

from tunix.sft.dpo.dpo_trainer import DpoTrainer, DpoTrainingConfig

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from aptax.llm import MiniGPT
from aptax.dataset import create_dataloader
from aptax.dataset import load_squad, QADataset

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
vocab_size = tokenizer.vocab_size

max_seq_len = 1024
embed_dim = 192
num_heads = 6
batch_size = 32
num_transformer_blocks = 6
num_epochs = 300

data = load_squad()
dataset = QADataset(data, max_seq_len, tokenizer)
dataloader, batches_per_epoch = create_dataloader(dataset, batch_size=batch_size, num_epochs=1)
total_steps = num_epochs * batches_per_epoch
warmup_steps = max(1, total_steps // 10)
print(f"Dataset size: {len(dataset)}")
print(f"Batches per epoch: {batches_per_epoch}")
print(f"Total training steps: {total_steps:,}")
print(f"Warmup steps: {warmup_steps:,}")

In [ ]:
model = MiniGPT(
    max_seq_len=max_seq_len,
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_transformer_blocks=num_transformer_blocks,
    rngs=nnx.Rngs(0),
)
checkpoint_path = Path.cwd() / "checkpoints" / "minigpt.orbax"
checkpointer = orbax.checkpoint.PyTreeCheckpointer()
cpu_device = jax.devices('cpu')[0]
cpu_sharding = SingleDeviceSharding(cpu_device)
restore_args = jax.tree_util.tree_map(
    lambda _: checkpoint.ArrayRestoreArgs(sharding=cpu_sharding),
    nnx.state(model)
)
nnx.state(model)
restored_state = checkpointer.restore(
    checkpoint_path,
    item=nnx.state(model),
    restore_args=restore_args)

nnx.update(model, restored_state)
print(model)

In [ ]:
def loss_fn(model, batch):
    inputs, targets, mask = batch
    logits = model(inputs)
    full_loss = optax.softmax_cross_entropy_with_integer_labels(logits, targets)
    masked_loss = full_loss * mask
    loss = jnp.sum(masked_loss) / jnp.sum(mask)
    return loss, logits


@nnx.jit
def train_step(model, optimizer, metrics, batch):
    inputs, targets, mask = batch
    grad_fn = nnx.value_and_grad(loss_fn, has_aux=True)
    (loss, logits), grads = grad_fn(
        model,
        batch,
    )
    accuracy = (logits.argmax(axis=-1) == targets).mean()

    metrics.update(loss=loss, accuracy=accuracy, logits=logits, labels=targets)
    optimizer.update(grads)


def eval_step(model, metrics, batch):
    inputs, targets, mask = batch
    loss, logits = loss_fn(model, batch)
    accuracy = (logits.argmax(axis=-1) == targets).mean()
    metrics.update(loss=loss, accuracy=accuracy, logits=logits, labels=targets)


lr_schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=5e-4,
    warmup_steps=warmup_steps,
    decay_steps=total_steps,
    end_value=5e-5,
)
optimizer = nnx.ModelAndOptimizer(
    model,
    optax.adamw(learning_rate=lr_schedule, weight_decay=0.01),
)
metrics = nnx.MultiMetric(
    loss=nnx.metrics.Average("loss"),
    accuracy=nnx.metrics.Average("accuracy"),
)

In [ ]:
step = 0
score_history = defaultdict(list)
early_stop = EarlyStopping(min_delta=1e-2, patience=5)
for epoch in range(num_epochs):
    epoch_history = defaultdict(list)
    train_batches, test_batches = train_test_split(
        list(range(batches_per_epoch)), test_size=0.2, random_state=epoch
    )
    batch_idx = 0
    for batch in (pbar := tqdm(dataloader, total=batches_per_epoch)):
        inputs = jnp.array(batch["inputs"]).T
        labels = jnp.array(batch["labels"]).T
        loss_mask = jnp.array(batch["loss_mask"]).T

        step_type = "train" if batch_idx in train_batches else "valid"
        if step_type == "train":
            train_step(model, optimizer, metrics, (inputs, labels, loss_mask))
        else:
            eval_step(model, metrics, (inputs, labels, loss_mask))

        for metric, value in metrics.compute().items():
            epoch_history[f"{step_type}_{metric}"].append(value)
        metrics.reset()

        current_lr = lr_schedule(step)
        pbar.set_description(
            f"epoch: {epoch + 1}/{num_epochs}, lr: {current_lr:.2e}, "
            f"loss (train/valid): {np.mean(epoch_history['train_loss']):.3f}/{np.mean(epoch_history['valid_loss']):.3f}, "
            f"acc (train/valid): {np.mean(epoch_history['train_accuracy']):.3f}/{np.mean(epoch_history['valid_accuracy']):.3f}"
        )
        step += 1
        batch_idx += 1

    for metric, values in epoch_history.items():
        score_history[metric].append(values)
    early_stop = early_stop.update(np.mean(score_history["train_loss"]))
    if early_stop.should_stop:
        print(f"\nMet early stopping criteria! Breaking at epoch {epoch}.")
        break